# Mosaic integration

Mosaic integration combines batches that share only some modalities. For example, a paired RNA + ATAC batch can link an RNA-only batch and an ATAC-only batch. Each method accepts one batch pattern, and every mosaic method reads ATAC as peaks. This tutorial runs StabMap and scMoMaT on `D46`, with 21,416 cells in three batches: RNA + ADT, RNA + ATAC, and RNA only.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_mosaic.ipynb).

## 1. Install

This cell installs `multibench-sc`. It keeps the numpy and pandas that are already installed, so Colab needs no restart.

<details>
<summary>Details</summary>

On Colab, choose a GPU runtime before you run the notebook: Runtime -> Change runtime type -> T4 GPU. On a CPU runtime, an environment that has a smaller CPU build gets that build, and training methods run slower.

</details>

In [ ]:
import numpy, pandas
%pip install -q multibench-sc numpy=={numpy.__version__} pandas=={pandas.__version__}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D46` (97 MB) once. Each method runs in its own environment. `mtb.env.install` downloads the environments for StabMap and scMoMaT, with no conda needed. The download is 1.8 GB on a computer without a GPU and 5.5 GB on one with a GPU.

<details>
<summary>Details</summary>

`state` is `PACKED` for an environment downloaded now and `have` for one that was already there. Without `dry_run=False`, `mtb.env.install` downloads nothing and returns the plan with its sizes.

Environments go to `mtb.config.DEFAULT.envs_dir`. To use another disk, set it before this cell.

From a terminal, `multibench env install --methods StabMap,scMoMaT --packed --run` does the same. With `--category mosaic` instead of `--methods`, it installs every mosaic environment: 6 envs, 15.6 GB to download on a CPU host, 20.8 GB on a GPU host.

</details>

In [ ]:
import pandas as pd
import multibench as mtb

mtb.data.fetch("D46")

In [ ]:
METHODS = ["StabMap", "scMoMaT"]
envs = mtb.env.install(METHODS, dry_run=False)
pd.DataFrame(envs)[["env", "methods", "state"]]

## 3. Run the methods

`run_all` runs each method on `D46` and scores its output with the scIB metrics. `res.summary` has one row per method with its status, run time and scores. Higher is better for every metric.

<details>
<summary>Details</summary>

Each method writes an embedding: a table of numbers with one row per cell. `run_all` scores it against the cell-type labels of `D46`.

The clustering metrics `ARI`, `NMI`, `ASW`, `iASW`, `iF1` and `cLISI` measure how well the embedding separates the cell types. The batch metrics `ASW_batch`, `GC` and `iLISI` measure how well the batches mix. They appear only when the data has several batches.

scMoMaT writes a graph instead of an embedding. `run_all` scores its UMAP, and its status reads `CHAIN_OK_GRAPH_METHOD`.

`res.failures` says why a method failed. `params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_batch("out/D46")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run_all("D46", "mosaic", methods=METHODS, out_dir="out/D46")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size shows the rank within a column, and bigger is better. The fill compares a value with the other rows in the same column.

<details>
<summary>Details</summary>

The lightest fill is the lowest value in this figure, not zero.

Metrics are grouped by family: blue for dimension reduction and clustering, green for batch correction. Each family starts with an `Overall` bar. Its length and colour both show the family score.

A column whose rows all hold the same value is drawn grey, and the note under the figure names it.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw counts, with one AnnData per batch and a cell type for each cell. `mtb.io.export_dataset` with `batch_index=` writes one batch per call. Number the batches to match a pattern that `mtb.describe_layout("mosaic")` lists, and give ATAC as peaks.

<details>
<summary>Details: export</summary>

`mtb.describe_layout("mosaic")` lists the batch patterns and the methods that accept each one. The command line writes one batch per call with `multibench convert ... --category mosaic --batch-index N`.

The folder name, `MYMOSAIC`, is the dataset name for `run_all`, and `data_path` is the folder that holds it.

`mtb.scan("MYMOSAIC", "mosaic", data_path="mydata")` checks the folder and the environments without running anything. Its `reason` column says what is missing.

`overwrite=True` replaces the files of an earlier run of this cell. Without it, `export_dataset` raises `FileExistsError` rather than replace a file.

</details>

Here the first 1,500 cells of each `D46` batch take the place of your data:

In [ ]:
d = mtb.config.DEFAULT.data_path / "D46"
n = 1500
rna = [mtb.io.read_canonical(d / f"rna{b}.h5")[:n] for b in (1, 2, 3)]
labels = [pd.read_csv(d / f"cty{b}.csv")["x"].values[:n] for b in (1, 2, 3)]
adt1 = mtb.io.read_canonical(d / "adt1.h5")[:n]
atac2 = mtb.io.read_canonical(d / "atac2.h5")[:n]

Write the folder, then run the same methods on it:

In [ ]:
out = "mydata/MYMOSAIC"
# batch 1: RNA + ADT
mtb.io.export_dataset(rna[0], out, adt=adt1, labels=labels[0],
                      batch_index=1, category="mosaic", overwrite=True)
# batch 2: RNA + ATAC peaks
mtb.io.export_dataset(rna[1], out, atac=atac2, atac_kind="peak", labels=labels[1],
                      batch_index=2, category="mosaic", overwrite=True)
# batch 3: RNA only
mtb.io.export_dataset(rna[2], out, labels=labels[2],
                      batch_index=3, category="mosaic", overwrite=True)

mine = mtb.run_all("MYMOSAIC", "mosaic", methods=METHODS, data_path="mydata",
                   out_dir="out/MYMOSAIC")
mine.summary

In [ ]:
mine.plot()

## 6. Stored scores

The package ships stored scores for 4 methods on `D45`. `load_results` reads them and `mtb.plot.bubble` draws them, without running anything.

<details>
<summary>Details</summary>

There are no stored scores for `D46`. `D45` is a larger mosaic dataset with another batch pattern.

`source="rerun"` reads the package's own runs of the methods. There is no published scIB table for mosaic, so `source="published"`, the default, raises `FileNotFoundError`.

The stored scores used the `leidenalg` backend for Leiden clustering, and `run_all` uses `igraph` by default. The two backends can move ARI by up to about 0.1. To compare your runs with these scores, set `mtb.config.DEFAULT.leiden_flavor = "leidenalg"` before `run_all`.

</details>

In [ ]:
long = mtb.load_results("mosaic", dataset="D45", source="rerun")
mtb.plot.bubble(long)

## Troubleshooting

`res.failures` lists each method that failed or was skipped. Its `error` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| `env.install` refuses on macOS or Windows | methods run only on Linux: use Colab or a Linux machine |
| a method needs an NVIDIA GPU | choose a GPU runtime on Colab, or a machine with a GPU |
| a warning that values are not whole numbers | export raw counts, for example with `rna="layer:counts"` |
| `... matrix/data as cells x features` | the matrix is transposed: export it again with `mtb.io.export_dataset` |
| a method times out | raise `timeout=` in `run_all` |
| low `label_order_confidence` | several label files fit the cell count: check `label_order_candidates` in `res.results` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- `mtb.cite(METHODS)` returns the citations for the benchmark and the methods you ran.
- The other tutorials: [vertical](https://dsichang.github.io/scMultiBench/tutorials/vertical/), [diagonal](https://dsichang.github.io/scMultiBench/tutorials/diagonal/), [cross](https://dsichang.github.io/scMultiBench/tutorials/cross/).
- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.